# Challenge:

- Build a synthetic dataset generator, write models that can generate datasets.
- Use a variety of models and prompts for diverse output, try quantizing and not quantizing, try different shapes and sizes.
- Create a Gradio UI for your product

## Idea

- Generate sample data for use when developing a REST API.

### Input: API spec, in different formats.

(API spec is already known vs suggest structure and endpoints based on model?)
  
- Empty JSON structure
- Java
- OAD (OpenAPI document) - in JSON or YAML
- UML? (Image or Mermaid chart def)
- URL - for example running Swagger UI instance
- As GUI - dropdowns or other UI elements to select entities and relations

In [12]:
import torch, accelerate, bitsandbytes, transformers

print(f"torch=={torch.__version__}")
print(f"accelerate=={accelerate.__version__}")
print(f"bitsandbytes=={bitsandbytes.__version__}")
print(f"transformers=={transformers.__version__}")

torch==2.6.0+cu124
accelerate==1.13.0
bitsandbytes==0.49.2
transformers==4.57.6


Installs in the correct order

```!uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124```

```!uv pip install --no-deps accelerate bitsandbytes transformers==4.57.6 huggingface_hub[hf_xet]``` 


In [13]:
# Imports

from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login

load_dotenv(override=True)

True

In [14]:
# Constants

LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
QWEN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PHI_MODEL = "microsoft/Phi-4-mini-instruct"
OPENAPI_FILE = "storefront-sample.json"

In [15]:
# Huggingface login

login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
# in case $ref is used in the schema - resolve_ref


def resolve_ref(spec, ref):
    path = ref.replace("#/", "").split("/")
    obj = spec
    for p in path:
        obj = obj[p]
    return obj

In [17]:
# Extraction function

import json


def extract_schema(openapi, path, method):
    method = method.lower()
    endpoint = openapi["paths"][path][method]
    schema = endpoint["requestBody"]["content"]["application/json"]["schema"]
    return schema


with open(OPENAPI_FILE) as f:
    spec = json.load(f)

schema = extract_schema(spec, "/cart/items", "post")

if "$ref" in schema:
    schema = resolve_ref(spec, schema["$ref"])

print(json.dumps(schema, indent=2))

{
  "type": "object",
  "required": [
    "product_id",
    "quantity"
  ],
  "properties": {
    "product_id": {
      "type": "string",
      "format": "uuid"
    },
    "quantity": {
      "type": "integer",
      "minimum": 1
    }
  }
}


In [18]:
user_prompt = f"""Generate realistic JSON objects following this schema:
{schema}
"""
system_prompt = """
You excel at generating realistic JSON objects following a given schema.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [19]:
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", quantization_config=quant_config
    )
    return model, tokenizer


def generate_json(model, tokenizer, messages, max_new_tokens=1000):
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to("cuda")

    print(f"  Input tokens: {inputs['input_ids'].shape[1]}")
    print(f"  Chat template found: {tokenizer.chat_template is not None}")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    print(f"  Output tokens: {len(new_tokens)}")
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [20]:
import re
from jsonschema import validate, ValidationError


def extract_json(text):
    """Pull the first complete JSON object or array out of model output."""
    if text is None:
        return None

    # Try ```json ... ``` code blocks first
    code_block = re.search(r"```(?:json)?\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```", text)
    if code_block:
        return code_block.group(1).strip()

    # Fall back to matching outermost braces/brackets
    for start, ch in enumerate(text):
        if ch not in "{[":
            continue
        close = "}" if ch == "{" else "]"
        depth = 0
        for end in range(start, len(text)):
            if text[end] == ch:
                depth += 1
            elif text[end] == close:
                depth -= 1
                if depth == 0:
                    return text[start : end + 1]
    return None


def validate_json(text):
    """Parse JSON string, return (parsed_object, is_valid) tuple."""
    try:
        return json.loads(text), True
    except (json.JSONDecodeError, TypeError):
        return None, False


def validate_schema(data, target_schema):
    """Check if data conforms to the target schema. Returns (conforms, error_message)."""
    items = data if isinstance(data, list) else [data]
    errors = []
    for i, item in enumerate(items):
        try:
            validate(instance=item, schema=target_schema)
        except ValidationError as e:
            errors.append(f"Item {i}: {e.message}")
    if errors:
        return False, "; ".join(errors)
    return True, None

In [21]:
import gc

MODELS = {
    "Llama 3.1 8B": LLAMA_MODEL,
    "Qwen 2.5 Coder 7B": QWEN_MODEL,
    "Phi 4 Mini": PHI_MODEL,
}

results = []

for label, model_name in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  {label} ({model_name})")
    print(f"{'=' * 60}")

    try:
        model, tokenizer = load_model(model_name)
        raw_output = generate_json(model, tokenizer, messages)

        extracted = extract_json(raw_output)
        parsed, is_valid = validate_json(extracted)
        conforms, schema_errors = (
            validate_schema(parsed, schema) if is_valid else (False, "No valid JSON")
        )

        results.append(
            {
                "model": label,
                "valid_json": is_valid,
                "schema_match": conforms,
                "schema_errors": schema_errors,
                "raw_output": raw_output,
                "extracted": extracted,
                "parsed": parsed,
            }
        )

        print(f"\nRaw output:\n{raw_output}")
        print(f"\nExtracted JSON:\n{extracted}")
        print(f"\nValid JSON: {is_valid}")
        print(f"Schema match: {conforms}")
        if schema_errors:
            print(f"Schema errors: {schema_errors}")

        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"\nError: {e}")
        results.append(
            {
                "model": label,
                "valid_json": False,
                "raw_output": None,
                "extracted": None,
                "parsed": None,
                "error": str(e),
                "schema_match": False,
                "schema_errors": str(e),
            }
        )


  Llama 3.1 8B (meta-llama/Meta-Llama-3.1-8B-Instruct)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  Input tokens: 105
  Chat template found: True
  Output tokens: 218

Raw output:
Here are five realistic JSON objects following the given schema:

```json
[
  {
    "product_id": "00000000-0000-0000-0000-000000000000",
    "quantity": 1
  },
  {
    "product_id": "12345678-1234-1234-1234-123456789012",
    "quantity": 5
  },
  {
    "product_id": "98765432-9876-9876-9876-987654321098",
    "quantity": 10
  },
  {
    "product_id": "fedcba98-fedc-fedc-fedc-fedcba987654",
    "quantity": 3
  },
  {
    "product_id": "12345678-1234-1234-1234-123456789012",
    "quantity": 7
  }
]
```

Note: The UUID format is not exactly as specified in the schema (which was 'uuid'), but the generated IDs are valid UUIDs.

Extracted JSON:
[
  {
    "product_id": "00000000-0000-0000-0000-000000000000",
    "quantity": 1
  },
  {
    "product_id": "12345678-1234-1234-1234-123456789012",
    "quantity": 5
  },
  {
    "product_id": "98765432-9876-9876-9876-987654321098",
    "quantity": 10
  },
  {
    "pro

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Input tokens: 84
  Chat template found: True
  Output tokens: 55

Raw output:
```json
{
  "product_id": "123e4567-e89b-12d3-a456-426614174000",
  "quantity": 3
}
```

Extracted JSON:
{
  "product_id": "123e4567-e89b-12d3-a456-426614174000",
  "quantity": 3
}

Valid JSON: True
Schema match: True

  Phi 4 Mini (microsoft/Phi-4-mini-instruct)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  Input tokens: 76
  Chat template found: True
  Output tokens: 659

Raw output:
Sure, here are some examples of realistic JSON objects following the given schema:

1.
```json
{
  "type": "object",
  "required": ["product_id", "quantity"],
  "properties": {
    "product_id": {
      "type": "string",
      "format": "uuid"
    },
    "quantity": {
      "type": "integer",
      "minimum": 1
    }
  },
  "data": {
    "product_id": "550e8400-e29b-44d3-a7da-5d5cfb6ad8f5",
    "quantity": 3
  }
}
```

2.
```json
{
  "type": "object",
  "required": ["product_id", "quantity"],
  "properties": {
    "product_id": {
      "type": "string",
      "format": "uuid"
    },
    "quantity": {
      "type": "integer",
      "minimum": 1
    }
  },
  "data": {
    "product_id": "3c4c6f7-4b3b-44c6-a7b7-7d5cfb6ad8f6",
    "quantity": 10
  }
}
```

3.
```json
{
  "type": "object",
  "required": ["product_id", "quantity"],
  "properties": {
    "product_id": {
      "type": "string",
      "format": "uui

In [22]:
print(f"\n{'=' * 60}")
print("  Model Comparison Summary")
print(f"{'=' * 60}\n")
print(f"  {'Model':<25} {'Valid JSON':<14} {'Schema Match'}")
print(f"  {'-' * 53}")
for r in results:
    print(f"  {r['model']:<25} {str(r['valid_json']):<14} {r['schema_match']}")
    if r.get("schema_errors"):
        print(f"    -> {r['schema_errors']}")


  Model Comparison Summary

  Model                     Valid JSON     Schema Match
  -----------------------------------------------------
  Llama 3.1 8B              True           True
  Qwen 2.5 Coder 7B         True           True
  Phi 4 Mini                True           False
    -> Item 0: 'product_id' is a required property
